# Template video stream with opencv and imutils

## Sources:
- [Speed up video reading](https://pyimagesearch.com/2017/02/06/faster-video-file-fps-with-cv2-videocapture-and-opencv/)
- [OpenCV: FFMPEG: tag 0x34363268/'h264' is not supported with codec](https://stackoverflow.com/questions/52932157/opencv-ffmpeg-tag-0x34363268-h264-is-not-supported-with-codec)

## Import modules

In [1]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path
import time


# import 3rd-party modules
import numpy as np
import cv2
from imutils.video import FileVideoStream
from imutils.video import FPS


# import local modules


## Define CONSTANTS & variables

In [4]:
WINDOW_NAME = "frame" # set window name
QUEUE_SIZE = 128 # set queue size (i.e. buffer)

# set output video codec
# use avc1 instead of h264 for mp4 video as tag 0x34363268/'h264' is not supported with codec and mp4 format
CODEC = "avc1"

## Define functions

## Run logic

In [7]:
# set input & output video path
video_path = Path("/Users/derrickvanfrausum/Documents/bouldering_annotated.mov")
out_path = f"{video_path.parent}/{video_path.stem}_processed.mp4"

# initialize video stream
# pass transform fct to process frame in file video stream thread
fvs = FileVideoStream(path=str(video_path), transform=None, queue_size=QUEUE_SIZE)

# get video stream parameters
video_n_frames = int(fvs.stream.get(cv2.CAP_PROP_FRAME_COUNT))
video_fps = fvs.stream.get(cv2.CAP_PROP_FPS)
video_width = int(fvs.stream.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(fvs.stream.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"number of frames = {video_n_frames}")
print(f"fps = {video_fps}")
print(f"video width = {video_width}")
print(f"video height = {video_height}")

# create a videoWriter object
fourcc = cv2.VideoWriter_fourcc(*'avc1')
out_video = cv2.VideoWriter(filename=out_path, fourcc=fourcc, fps=video_fps, frameSize=(video_width, video_height))

# start file video stream thread and allow buffer to
# start to fill
print("[INFO] starting video file thread...")
fvs.start()
time.sleep(1.0)

# start fps timer to monitor runtime
fps = FPS().start()

# loop over frames from video file stream
while fvs.more():

	# grab frame from the threaded video file stream
	frame = fvs.read()
	if frame is None:
		print("Ignoring empty camera frame.")
		break

	# process frame

	# show frame
	cv2.imshow(WINDOW_NAME, frame)

    # wait for a key 
    # 0xFF to check what key we pressed on the keyboard
	key = cv2.waitKey(10) & 0xFF

    # break out of stream loop if esc or 'q' is pressed
	if key == 27 or key == ord('q'):        
		break
	
	# write output frame
	out_video.write(frame)

	# update fps counter
	fps.update()
	
# stop timer and display FPS information
fps.stop()
print("[INFO] elasped time: {:.2f}".format(fps.elapsed()))
print("[INFO] approx. FPS: {:.2f}".format(fps.fps()))

# close windows
cv2.destroyAllWindows()
cv2.waitKey(1) # workaround to effectively close window on mac

# release video stream & video rendering
fvs.stop()
out_video.release()

number of frames = 270
fps = 29.53724975385625
video width = 352
video height = 640
[INFO] starting video file thread...
Ignoring empty camera frame.
[INFO] elasped time: 3.53
[INFO] approx. FPS: 76.57
